In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Body Fat Percentage Estimation Using Unsupervised Learning",
    "",
    "This notebook estimates body fat percentage from a folder of unlabeled body images using a CNN-based autoencoder for feature extraction and K-Means clustering to group similar body types. The clusters can be mapped to body fat percentage ranges with domain knowledge."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1: Import Libraries"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import tensorflow as tf\n",
    "from tensorflow.keras import layers, models\n",
    "from sklearn.cluster import KMeans\n",
    "from PIL import Image\n",
    "import glob\n",
    "import matplotlib.pyplot as plt\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2: Load and Preprocess Images",
    "",
    "Load images from a folder, resize them to 128x128 pixels, and normalize pixel values to [0, 1]. Replace `folder_path` with the path to your image folder."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def load_images(folder_path):\n",
    "    images = []\n",
    "    for img_path in glob.glob(f\"{folder_path}/*.jpg\"):\n",
    "        img = Image.open(img_path).resize((128, 128))\n",
    "        img_array = np.array(img)\n",
    "        if img_array.shape == (128, 128, 3):  # Ensure RGB format\n",
    "            images.append(img_array)\n",
    "    images = np.array(images).astype('float32') / 255.0  # Normalize\n",
    "    return images\n",
    "\n",
    "# Replace with your folder path\n",
    "folder_path = \"path/to/your/image/folder\"\n",
    "images = load_images(folder_path)\n",
    "print(f\"Loaded {len(images)} images with shape {images.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3: Build CNN Autoencoder",
    "",
    "Create a CNN autoencoder to extract latent features from images."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def build_autoencoder():\n",
    "    input_shape = (128, 128, 3)\n",
    "    \n",
    "    # Encoder\n",
    "    encoder_input = layers.Input(shape=input_shape)\n",
    "    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_input)\n",
    "    x = layers.MaxPooling2D((2, 2), padding='same')(x)\n",
    "    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)\n",
    "    x = layers.MaxPooling2D((2, 2), padding='same')(x)\n",
    "    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)\n",
    "    encoded = layers.MaxPooling2D((2, 2), padding='same')(x)\n",
    "    \n",
    "    # Decoder\n",
    "    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(encoded)\n",
    "    x = layers.UpSampling2D((2, 2))(x)\n",
    "    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)\n",
    "    x = layers.UpSampling2D((2, 2))(x)\n",
    "    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)\n",
    "    x = layers.UpSampling2D((2, 2))(x)\n",
    "    decoded = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)\n",
    "    \n",
    "    # Autoencoder model\n",
    "    autoencoder = models.Model(encoder_input, decoded)\n",
    "    \n",
    "    # Encoder model for feature extraction\n",
    "    encoder = models.Model(encoder_input, encoded)\n",
    "    \n",
    "    autoencoder.compile(optimizer='adam', loss='mse')\n",
    "    return autoencoder, encoder"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4: Train Autoencoder and Extract Features",
    "",
    "Train the autoencoder and use the encoder to extract latent features."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def train_and_extract_features(images, epochs=10):\n",
    "    autoencoder, encoder = build_autoencoder()\n",
    "    autoencoder.fit(images, images, epochs=epochs, batch_size=32, verbose=1)\n",
    "    latent_features = encoder.predict(images)\n",
    "    latent_features = latent_features.reshape(len(images), -1)  # Flatten for clustering\n",
    "    return latent_features\n",
    "\n",
    "latent_features = train_and_extract_features(images, epochs=10)\n",
    "print(f\"Extracted latent features with shape {latent_features.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5: Cluster Latent Features",
    "",
    "Use K-Means to cluster the latent features into groups, which may correspond to body fat percentage ranges."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def cluster_features(latent_features, n_clusters=5):\n",
    "    kmeans = KMeans(n_clusters=n_clusters, random_state=42)\n",
    "    clusters = kmeans.fit_predict(latent_features)\n",
    "    return clusters\n",
    "\n",
    "n_clusters = 5\n",
    "clusters = cluster_features(latent_features, n_clusters)\n",
    "print(\"Cluster assignments for each image:\", clusters)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6: Visualize Results",
    "",
    "Display a few images from each cluster to inspect the groupings."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def visualize_clusters(images, clusters, n_clusters):\n",
    "    plt.figure(figsize=(15, 5))\n",
    "    for cluster_id in range(n_clusters):\n",
    "        cluster_indices = np.where(clusters == cluster_id)[0]\n",
    "        if len(cluster_indices) > 0:\n",
    "            plt.subplot(1, n_clusters, cluster_id + 1)\n",
    "            plt.imshow(images[cluster_indices[0]])\n",
    "            plt.title(f\"Cluster {cluster_id}\")\n",
    "            plt.axis('off')\n",
    "    plt.show()\n",
    "\n",
    "visualize_clusters(images, clusters, n_clusters)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 7: Map Clusters to Body Fat Ranges",
    "",
    "Map clusters to hypothetical body fat percentage ranges. This requires domain knowledge or manual labeling."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "body_fat_ranges = [\"<15%\", \"15-20%\", \"20-25%\", \"25-30%\", \">30%\"]\n",
    "for i, cluster in enumerate(clusters):\n",
    "    print(f\"Image {i+1} assigned to cluster {cluster} (Estimated body fat: {body_fat_ranges[cluster]})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Notes",
    "",
    "- **Image Folder**: Replace `folder_path` with the actual path to your image folder.\n",
    "- **Cluster Interpretation**: Clusters may not directly map to body fat percentages. Validate with domain expertise or label a subset of images.\n",
    "- **Tuning**: Adjust `n_clusters`, `epochs`, or the autoencoder architecture based on your dataset.\n",
    "- **Visualization**: The visualization shows one representative image per cluster. Inspect multiple images per cluster for better understanding."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}